# Simulating and decoding under noise

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do noise models, decoders, and confidence filtering determine logical error and shot rejection rates?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Configure depolarizing and reset noise on simulator primitives with `NoiseConfig`.
- Explain why a distance-two code requires readout confidence filtering (`max_readout_score`) to suppress syndrome faults.
- Compare logical error rates and shot rejection rates across physical fault probabilities with `deq`.
- Contrast distance-two error detection (`C4`) with distance-three active error correction (`[[7, 1, 3]]` Steane code).

## Setting up

The cell below loads the distance-two `C4` protocol and compiles six logical Q# benchmarks (`SPAM`, `cx_all`, and `syndrome` in both the $Z$ and $X$ bases).

In [ ]:
from _unit import (
    compile_c4_benchmarks,
    make_upgraded_protocol,
    plot_sweep,
    sample_sweep,
)

protocol = make_upgraded_protocol()
benchmarks = compile_c4_benchmarks()
print("Compiled benchmarks:", list(benchmarks))

## Configuring physical gate noise

> **Warning — Current `NoiseConfig` limitation (evolving interface):** In the current QDK release, `qdk.simulation.NoiseConfig` attaches noise to the **simulator's built-in QIR primitives** (`h`, `x`, `y`, `z`, `cx`, `mresetz`), **not** directly to bottom-layer `qodec` instruction mnemonics. As a consequence, two distinct bottom-layer instructions that lower to the same primitive share its noise channel, and `noise.mresetz.x = p` injects a post-reset $X$ flip **after** the measurement bit is recorded (so pure `M` readout assignment flips are not parameterized separately from pre-measurement gate noise). Instruction-level noise configuration aligned directly with `qodec` layers is planned for an upcoming release and will replace or extend this interface.

With that distinction in mind, bottom-layer operations map onto `NoiseConfig` primitives as follows:

| Bottom-layer instruction | Simulator primitive | `NoiseConfig` setting |
| :--- | :--- | :--- |
| `R` (reset to $\vert 0\rangle$) | `mresetz` | `noise.mresetz.x = p` (applies $X$ after reset) |
| `H`, `X`, `Y`, `Z` | `h`, `x`, `y`, `z` | `gate.set_depolarizing(p)` ($p/3$ per single-qubit Pauli) |
| `CX` | `cx` | `noise.cx.set_depolarizing(p)` ($p/15$ per non-identity two-qubit Pauli) |
| `M` ($Z$ measurement) | `mresetz` | `noise.mresetz.x = p` acts **after** the measurement bit is recorded |

In our `C4` gadgets, depolarizing faults on the `H` and `CX` gates preceding `M` still corrupt ancilla and data measurement outcomes at rate $O(p)$.

To decode circuit-level faults with the optional `deq` package, pass `decoders.prepare_deq_decoder` (on platforms without `deq-runtime` wheels, such as Windows Arm64, the helper falls back to QDK's built-in `decoders.prepare_syndrome_decoder`, using nonzero-syndrome shot rejection when `max_readout_score` is set):

In [ ]:
from functools import partial
import importlib.util
from qdk.simulation import NoiseConfig, decoders, run_qir

if importlib.util.find_spec("deq") is not None:
    from deq.runtime import Runtime

    def make_runtime(seed: int) -> Runtime:
        return Runtime(
            decoder="black-box-relay-bp",
            decoder_config={"parallel": 1, "seed": seed},
            coordinator="monolithic",
            controller="jit",
        )

    sample_decoder = partial(decoders.prepare_deq_decoder, runtime_factory=make_runtime)
else:
    sample_decoder = decoders.prepare_syndrome_decoder

example_noise = NoiseConfig()
for gate in (example_noise.h, example_noise.cx):
    gate.set_depolarizing(0.01)
example_noise.mresetz.x = 0.01

sample_rows = run_qir(
    benchmarks["SPAM Z"],
    qodec=protocol,
    decoder=sample_decoder,
    noise=example_noise,
    type="stabilizer",
    shots=8,
    seed=42,
    on_shot_failure="discard",
)
print("Sample noisy shots (flags, logical_results):", sample_rows[:5])

## Why preparation flags alone leave first-order logical errors

Let's sweep physical fault probabilities $p \in \{2\times 10^{-3}, 5\times 10^{-3}, 10^{-2}\}$ while rejecting **only** shots where a preparation `reject` flag fired (`max_readout_score=None`):

In [ ]:
fault_probabilities = [2e-3, 5e-3, 1e-2]

sweep_flags_only = sample_sweep(
    protocol,
    benchmarks,
    fault_probabilities,
    shots=600,
    workers=4,
    max_readout_score=None,
)
plot_sweep(sweep_flags_only, title="C4: preparation flags only (pooled X/Z)")

Look at the left-hand plot: **all three curves (`SPAM`, `cx_all`, and `syndrome`) still sit above the dashed $p_L = p$ break-even line with first-order $O(p)$ scaling!** Why didn't the preparation `reject` flag put `SPAM` below break-even?

The preparation `reject` flag (`M 4`) only heralds **weight-two spreading faults** on control qubit `0`. Any **single-qubit fault** on qubits `1..3` (or inside `cx_all`, `syndrome`, or `measure_*_all`) leaves the `reject` flag equal to `0` and instead flips a **parity check** (such as `M0 ⊕ M1 ⊕ M2 ⊕ M3 = 1` during measurement). Because a distance-$d$ code detects up to $d - 1$ errors yet uniquely corrects only $\lfloor (d - 1)/2 \rfloor$, `C4` ($d = 2$) detects that a single fault occurred without knowing *which* qubit failed—forcing the decoder to guess and leaving an $O(p)$ logical error rate across every curve.

## Rejecting low-confidence shots with readout scores

`deq` assigns each decoded logical readout a relative confidence score in `[0, 1]` comparing its chosen correction against the best alternative that flips that readout (`0.0` means unambiguous; near `0.5` means a tie; scores are not calibrated failure probabilities). Passing `max_readout_score=0.05` rejects shots with ambiguous syndromes without changing a single line of the `qodec` or QIR:

In [ ]:
sweep_postselected = sample_sweep(
    protocol,
    benchmarks,
    fault_probabilities,
    shots=1500,
    workers=4,
    max_readout_score=0.05,
)
plot_sweep(
    sweep_postselected,
    title="C4: preparation flags + readout score <= 0.05 (pooled X/Z)",
)

With `max_readout_score=0.05`, **all three curves (`SPAM`, `cx_all`, and `syndrome`) plunge strictly below the $p_L = p$ break-even line** with quadratic $O(p^2)$ error suppression—trading a higher shot-rejection rate on the right plot for much cleaner accepted shots.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why do all three C4 curves sit above break-even with preparation flags alone, yet drop below break-even when readout-score filtering is enabled?</summary>

<div style="padding:0.1em 1em;">

Preparation `reject` flags only catch weight-two spreading faults during state preparation. Single-qubit faults flip stabilizer `checks` (in `syndrome` and `measure_*_all`), which a distance-two code can detect but cannot uniquely correct; enabling `max_readout_score=0.05` discards those ambiguous single-fault shots so only $O(p^2)$ multi-fault events remain.

</div>

</details></div>

## From error detection to active error correction with the Steane code

What if you want active single-fault correction without tuning a `max_readout_score` threshold? Moving from distance-two `C4` to the **distance-three $[[7, 1, 3]]$ Steane code** ($\lfloor (3 - 1)/2 \rfloor = 1$) and building with `strategy="flagged-css/v1"` gives every gadget (`prepare_z_all`, `syndrome`, and `measure_z_all`) certified **gadget distance `3`**, allowing the decoder to correct single faults directly:

In [ ]:
import qodec
from qdk import Result, TargetProfile, ec, qsharp

steane = qodec.Code(
    "Steane",
    stabilizers=[
        "X_0 X_2 X_4 X_6", "X_1 X_2 X_5 X_6", "X_3 X_4 X_5 X_6",
        "Z_0 Z_2 Z_4 Z_6", "Z_1 Z_2 Z_5 Z_6", "Z_3 Z_4 Z_5 Z_6",
    ],
    x=["X_0 X_1 X_2 X_3 X_4 X_5 X_6"],
    z=["Z_0 Z_1 Z_2 Z_3 Z_4 Z_5 Z_6"],
)
steane_proto = ec.build_qodec(steane, strategy="flagged-css/v1", strict=False)
for name in ("prepare_z_all", "syndrome", "measure_z_all"):
    dist = ec.GadgetProfile(steane_proto.layers[0].gadgets[name]).distance()
    print(f"Steane {name:<14} -> gadget distance {dist}")

qsharp.init(target_profile=TargetProfile.Adaptive_RI)
qsharp.eval("""
operation prepare_z_all(b : Qubit) : Unit { body intrinsic; }
@Measurement() operation measure_z_all(b : Qubit) : Result { body intrinsic; }
operation syndrome(b : Qubit) : Unit { body intrinsic; }

operation SteaneSyndromeZ() : Result {
    use b = Qubit();
    prepare_z_all(b);
    syndrome(b);
    measure_z_all(b)
}
""")
steane_qir = str(qsharp.compile("SteaneSyndromeZ()"))
steane_dec = sample_decoder

for p_val in [1e-3, 2e-3, 5e-3]:
    noise = NoiseConfig()
    for g in ("h", "x", "y", "z", "cx"):
        getattr(noise, g).set_depolarizing(p_val)
    noise.mresetz.x = p_val
    shots = run_qir(
        steane_qir,
        qodec=steane_proto,
        decoder=steane_dec,
        noise=noise,
        type="stabilizer",
        shots=1000,
        seed=42,
    )
    wrong = sum(r != Result.Zero for r in shots)
    ler, acc = wrong / len(shots), len(shots) / 1000
    print(f"p={p_val:.3f} -> accepted={len(shots)}/1000 ({acc:.1%}), active-EC LER={ler:.4f}")

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why can the distance-three Steane protocol suppress single-fault logical errors without readout-score filtering?</summary>

<div style="padding:0.1em 1em;">

With code distance $d = 3$ ($\lfloor (3 - 1)/2 \rfloor = 1$) and `flagged-css/v1` gadgets of certified gadget distance `3`, any single circuit fault produces a check pattern that uniquely identifies its logical effect, letting the decoder correct it instead of discarding the shot.

</div>

</details></div>

## Exercise: Verify error suppression with confidence filtering

Complete `is_postselected_below_physical` so it samples `syndrome Z` on `C4` at $p = 0.005$ with `max_readout_score=0.05` and returns `True` if the conditional logical error rate (`wrong / accepted`) is strictly below `p_val`.

In [ ]:
from _unit import (
    compile_c4_benchmarks,
    exercise,
    make_upgraded_protocol,
    sample_sweep,
)


@exercise
def is_postselected_below_physical() -> bool:
    protocol = make_upgraded_protocol()
    syndrome_z = {'syndrome Z': compile_c4_benchmarks()['syndrome Z']}
    sweep = sample_sweep(
        protocol, syndrome_z, [5e-3], shots=400, workers=2, max_readout_score=0.05
    )
    p_val, counts = sweep['syndrome Z'][0]
    # Compute ler = counts['wrong'] / counts['accepted'] and return bool(ler < p_val)
    return ...

**Hint**

Compute `ler = counts['wrong'] / counts['accepted']` and return `bool(ler < p_val)`.

In [ ]:
from _unit import (
    compile_c4_benchmarks,
    exercise,
    make_upgraded_protocol,
    sample_sweep,
)


@exercise
def is_postselected_below_physical() -> bool:
    protocol = make_upgraded_protocol()
    syndrome_z = {'syndrome Z': compile_c4_benchmarks()['syndrome Z']}
    sweep = sample_sweep(
        protocol, syndrome_z, [5e-3], shots=400, workers=2, max_readout_score=0.05
    )
    p_val, counts = sweep['syndrome Z'][0]
    ler = counts['wrong'] / counts['accepted']
    return bool(ler < p_val)

`True`. Rejecting shots with `max_readout_score > 0.05` discards single-fault syndromes on `C4`, suppressing the conditional logical error rate from $O(p)$ to $O(p^2) \ll 0.005$.

## Further reading

- [Configure simulator noise and select a decoder ↗](https://learn.microsoft.com/en-us/azure/quantum/run-decode-logical-programs#configure-simulator-noise)
- [Reject low-confidence deq results ↗](https://learn.microsoft.com/en-us/azure/quantum/run-decode-logical-programs#reject-low-confidence-deq-results)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../08-openqasm-integration/openqasm_integration.workbook.ipynb" title="Previous unit: Calling Qodec Instructions from OpenQASM" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Calling Qodec Instructions from OpenQASM</span></span></a><a href="../10-capstone/capstone.workbook.ipynb" title="Next unit: Capstone: Custom Gadgets and Hardware Noise" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Capstone: Custom Gadgets and Hardware Noise</span></span><span aria-hidden="true">&#8250;</span></a></div>